# Study 874 — IPO-Price Anchoring — the teardown

The Fama-MacBeth anchoring slope, the below-offer basket spread, the 1,000-permutation placebo, the two-era cut, the costed timer, and the 20-seed synthetic control. Real-tape numbers are the frozen headline (`docs/results.md`).

In [1]:
R = {'as_of': '2026-06-30', 'fingerprint': '1eaa178051af', 'n_names': 44, 'n_ipo': 40, 'n_direct': 4, 'start': '2014-01-31', 'end': '2026-06-30', 'active_months': 86, 'n_obs': 2698, 'avg_names': 30.3, 'below_share': 42.0, 'anchor_slope': -0.0023, 'anchor_bps10': -2.3, 'anchor_t': -0.39, 'anchor_t1s': -0.4, 'anchor_share_neg': 44, 'anchor_n': 86, 'anchor_slope_ipo': -0.0018, 'anchor_t_ipo': -0.3, 'below_bps': -56.84, 'below_ann': -6.61, 'below_t': -0.56, 'below_welch': -0.35, 'below_leg': 13.07, 'above_leg': 69.91, 'below_n': 76, 'below_bps_ipo': -99.93, 'below_t_ipo': -0.94, 'plac_obs': -0.00233, 'plac_mean': -0.00026, 'plac_sd': 0.00478, 'plac_p_left': 0.322, 'plac_p_two': 0.61, 'plac_draws': 1000, 'era_early_bps': -43.31, 'era_early_t': -0.18, 'era_early_n': 29, 'era_late_bps': -65.19, 'era_late_t': -0.87, 'era_late_n': 47, 'timer': [(10.0, 3.0, 56.84, 0.56, 11.84, 1.43, 0.12, -57.0), (20.0, 5.0, 56.84, 0.56, -24.82, -2.94, -0.24, -61.7)], 'ctrl': [(0.0, 0.0023, 0.41, 0), (0.15, -0.1441, -12.54, 100), (0.3, -0.2946, -19.76, 100)]}

## Data stamp

In [2]:
print(f"{R['n_names']} curated listings ({R['n_ipo']} IPOs + {R['n_direct']} direct), "
      f"vs SPY, {R['start']} -> {R['end']}")
print(f"{R['active_months']} active months, {R['n_obs']} name-months, "
      f"avg {R['avg_names']:.1f} names/month, below-offer share {R['below_share']:.1f}%")
print(f"as-of {R['as_of']}   fingerprint {R['fingerprint']}")

44 curated listings (40 IPOs + 4 direct), vs SPY, 2014-01-31 -> 2026-06-30
86 active months, 2698 name-months, avg 30.3 names/month, below-offer share 42.0%
as-of 2026-06-30   fingerprint 1eaa178051af


## Test 1 — the anchoring pull (Fama-MacBeth cross-sectional slope, HAC t)

Forward market-adjusted return regressed on `gap = log(price/offer)` each month; average the monthly slopes; NW(6) *t*. Anchoring ⇒ negative slope.

In [3]:
print(f"all listings: slope {R['anchor_slope']:+.4f} ({R['anchor_bps10']:+.1f} bps/mo "
      f"per +10% above offer)  NW t = {R['anchor_t']:+.2f}  "
      f"({R['anchor_share_neg']}% of months negative, n={R['anchor_n']})")
print(f"IPOs only  : slope {R['anchor_slope_ipo']:+.4f}  NW t = {R['anchor_t_ipo']:+.2f}")

all listings: slope -0.0023 (-2.3 bps/mo per +10% above offer)  NW t = -0.39  (44% of months negative, n=86)
IPOs only  : slope -0.0018  NW t = -0.30


## Test 2 — the below-offer drag (below − above basket spread, HAC t)

In [4]:
print(f"all listings: {R['below_bps']:+.2f} bps/mo ({R['below_ann']:+.2f}%/yr)  "
      f"NW t = {R['below_t']:+.2f}  (Welch t = {R['below_welch']:+.2f}, n={R['below_n']})")
print(f"   below-offer basket {R['below_leg']:+.2f} vs above-offer basket {R['above_leg']:+.2f} bps/mo")
print(f"IPOs only  : {R['below_bps_ipo']:+.2f} bps/mo  NW t = {R['below_t_ipo']:+.2f}")

all listings: -56.84 bps/mo (-6.61%/yr)  NW t = -0.56  (Welch t = -0.35, n=76)
   below-offer basket +13.07 vs above-offer basket +69.91 bps/mo
IPOs only  : -99.93 bps/mo  NW t = -0.94


## Placebo — shuffle gap→forward-return within each month (1,000 permutations)

In [5]:
print(f"observed slope {R['plac_obs']:+.5f} vs placebo mean {R['plac_mean']:+.5f} "
      f"(sd {R['plac_sd']:.5f}) over {R['plac_draws']:,} draws")
print(f"left-tail p = {R['plac_p_left']:.3f}   two-sided p = {R['plac_p_two']:.3f}  "
      f"-> indistinguishable from a random pairing")

observed slope -0.00233 vs placebo mean -0.00026 (sd 0.00478) over 1,000 draws
left-tail p = 0.322   two-sided p = 0.610  -> indistinguishable from a random pairing


## Robustness — below-offer spread, two eras (split 2022-07)

In [6]:
print(f"pre-2022-07: {R['era_early_bps']:+.2f} bps/mo  NW t = {R['era_early_t']:+.2f} (n={R['era_early_n']})")
print(f"2022-07 on : {R['era_late_bps']:+.2f} bps/mo  NW t = {R['era_late_t']:+.2f} (n={R['era_late_n']})")
print('same (negative) sign both halves, insignificant in each -> one correlated cohort')

pre-2022-07: -43.31 bps/mo  NW t = -0.18 (n=29)
2022-07 on : -65.19 bps/mo  NW t = -0.87 (n=47)
same (negative) sign both halves, insignificant in each -> one correlated cohort


## The timer — SHORT below-offer / LONG above-offer, net of borrow + costs

In [7]:
for cb, br, g, gt, n, na, nt, dd in R['timer']:
    print(f"cost={cb:>4.1f} bps, borrow={br:.0f}%/yr: gross {g:+.2f} (t={gt:+.2f}) "
          f"-> net {n:+.2f} bps/mo ({na:+.2f}%/yr, t={nt:+.2f})  max DD {dd:.1f}%")

cost=10.0 bps, borrow=3%/yr: gross +56.84 (t=+0.56) -> net +11.84 bps/mo (+1.43%/yr, t=+0.12)  max DD -57.0%
cost=20.0 bps, borrow=5%/yr: gross +56.84 (t=+0.56) -> net -24.82 bps/mo (-2.94%/yr, t=-0.24)  max DD -61.7%


## Synthetic positive control — the machinery is unbiased

Live: the FM detector must NOT fire on the null and must recover a planted pull.

In [8]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # the study package
sys.path.insert(0, os.path.abspath("../../.."))    # repo root
import numpy as np
from ipo_anchor import data, strategy as st
for edge in (0.0, 0.15):
    r = st.synthetic_control(edge, n_seeds=20)
    print(f"edge={edge:.2f}: mean slope {r['mean_slope']:+.4f}  mean HAC t = {r['mean_t']:+.2f}  "
          f"|t|>=2 rejection rate = {r['reject_rate']*100:.0f}%")

edge=0.00: mean slope +0.0023  mean HAC t = +0.41  |t|>=2 rejection rate = 0%


edge=0.15: mean slope -0.1441  mean HAC t = -12.54  |t|>=2 rejection rate = 100%


## Verdict

- **Signal — None.** Anchoring-pull FM slope **-0.0023** (NW *t* = **-0.39**, placebo two-sided *p* = 0.61); below-offer drag **-56.8 bps/mo** (NW *t* = **-0.56**), same sign in both eras (*t* = -0.18 / -0.87) but insignificant throughout. Both point the claimed way; neither clears |t| ≥ 2 — an underpowered, one-cohort curated cross-section. The 20-seed synthetic control fires on 0/20 nulls and 100% on a planted pull, so the flatness is real, not machinery.
- **Tradability — Mirage.** Short-below / long-above earns +56.8 bps/mo gross at *t* = +0.56; borrow + costs take net to +11.8 → -24.8 bps/mo on a ~60% drawdown.